In [3]:
import sys
import os
from pathlib import Path

# add parent directory to path to fix imports
parent_dir = Path().resolve().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from baseline_vectors.grad_cam import GradCAM
from embedding_pipeline.rcnn import run_detector, postprocess_detections
from embedding_pipeline.stft import stft
from embedding_pipeline.embedder import Embedder

import torch
from cnn.resnet import ResNet

In [4]:
def collect_labeled_files(root_dir, label, max_count=None):
    samples = []
    for root, _, files in os.walk(root_dir):
        for file in files:
            file_path = os.path.join(root, file)
            samples.append((file_path, label))
            if max_count is not None and len(samples) >= max_count:
                return samples
    return samples

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# resnet for grad cam
model = ResNet().to(device)
model.load_state_dict(torch.load("../cnn/resnet_checkpoint_7.pth", map_location=device)["model"])

# embedder for dense-map embeddings
embedding_dim = 128
embedder = Embedder(embedding_dim=embedding_dim,).to(device)
embedder.load_state_dict(torch.load("../embedding_pipeline/isotropic_embedder_checkpoint.pth", map_location=device))
embedder.eval()

RuntimeError: Error(s) in loading state_dict for Embedder:
	Missing key(s) in state_dict: "conv1.0.weight", "conv1.0.bias", "conv1.1.weight", "conv1.1.bias", "conv1.1.running_mean", "conv1.1.running_var", "conv2.0.weight", "conv2.0.bias", "conv2.1.weight", "conv2.1.bias", "conv2.1.running_mean", "conv2.1.running_var", "conv3.0.weight", "conv3.0.bias", "conv3.1.weight", "conv3.1.bias", "conv3.1.running_mean", "conv3.1.running_var", "conv4.0.weight", "conv4.0.bias", "conv4.1.weight", "conv4.1.bias", "conv4.1.running_mean", "conv4.1.running_var", "fc1.0.weight", "fc1.0.bias", "fc1.1.weight", "fc1.1.bias", "fc1.1.running_mean", "fc1.1.running_var", "fc2.weight", "fc2.bias". 
	Unexpected key(s) in state_dict: "epoch", "model_state_dict", "embedding_dim", "optimizer_state_dict", "train_losses", "val_losses", "isotropy_penalties". 

In [ ]:
max_files = 200

fake_files = collect_labeled_files(
    "../data/release_in_the_wild/train/fake", label=1, max_count=max_files
)
real_files = collect_labeled_files(
    "../data/release_in_the_wild/train/real", label=0, max_count=max_files
)

Total dataset files: 400
Example item: ('../data/release_in_the_wild/train/real\\10330.wav', 0)


In [ ]:
# dataset CAM vector generation:
# fake clips: stft -> cam -> detector -> isotropic dense map -> map crop -> pooled vector
# real clips: stft -> isotropic dense map -> keep every dense-map location vector
import numpy as np
import torch
from pathlib import Path

max_time_windows = 256
batch_size = 32

all_vectors = []
all_labels = []
all_metadata = []
skipped_files = []

model.eval()
embedder.eval()

for file_idx, (file_path, label) in enumerate(real_files):
    batch = []
    if (file_idx + 1) % batch_size != 0:  
        magnitude_db, _ = stft(file_path)
        batch.append(torch.tensor(magnitude_db, dtype=torch.float32).unsqueeze(0))
    else:
        source_h, source_w = int(batch[0].shape[1]), int(batch[0].shape[2])
        torch.stack(batch, dim=0)  # shape: (batch_size, 1, freq_bins, time_windows)

        with torch.no_grad():
            embeddings = embedder(batch)

        for embedding in embeddings:
            all_vectors.append(embedding)
            all_labels.append(int(label))
            all_metadata.append({
                "file_path": file_path,
                "label": int(label),
                })

    if (file_idx + 1) % batch_size == 0:
        print(f"Processed files: {file_idx + 1}/{len(real_files)} | saved vectors so far: {len(all_vectors)}")


grad_cam = GradCAM(model)
for file_idx, (file_path, label) in enumerate(fake_files):
    input_tensor = torch.tensor(magnitude_db, dtype=torch.float32).unsqueeze(0).to(device)
    cam = grad_cam.generate(input_tensor)
    grad_cam.remove_hooks()

    cam_tensor = torch.from_numpy(np.asarray(cam, dtype=np.float32)).unsqueeze(0).repeat(3, 1, 1)
    raw_dets = run_detector(cam_tensor, device=device)
    filtered_dets = postprocess_detections(
        raw_dets,
        cam_tensor.shape,
        conf_thresh=0.1,
        max_detections=100,
    )

    if not filtered_dets:
        continue

    with torch.no_grad():
        feature_map = embedder.forward_features(input_tensor)
        h_feat, w_feat = feature_map.shape[-2], feature_map.shape[-1]
        dense_embeddings = embedder(input_tensor)[0].view(h_feat, w_feat, -1)

    src_h, src_w = int(input_tensor.shape[-2]), int(input_tensor.shape[-1])

    # for each detected box, find the corresponding location in the dense embedding map and save that vector
    for det in filtered_dets:
        x1, y1, x2, y2 = det["box"]
        cx = 0.5 * (x1 + x2)
        cy = 0.5 * (y1 + y2)

        gx = int(round(cx * (w_feat - 1) / max(src_w - 1, 1)))
        gy = int(round(cy * (h_feat - 1) / max(src_h - 1, 1)))
        gx = max(0, min(gx, w_feat - 1))
        gy = max(0, min(gy, h_feat - 1))

        selected_embedding = dense_embeddings[gy, gx]
        all_vectors.append(selected_embedding.detach().cpu())
        all_labels.append(int(label))
        all_metadata.append({
            "file_path": file_path,
            "label": int(label),
        })

    if (file_idx + 1) % batch_size == 0:
        print(f"Processed files: {file_idx + 1}/{len(fake_files)} | saved vectors so far: {len(all_vectors)}")
        
        
detected_vector_tensor = torch.stack(all_vectors, dim=0)
label_tensor = torch.tensor(all_labels, dtype=torch.long)

save_dict = {
    "vectors": detected_vector_tensor,
    "labels": label_tensor,
    "metadata": all_metadata,
    "label_map": {"real": 0, "fake": 1},
}

save_path = Path("detected_vectors_with_metadata_big.pt")
torch.save(save_dict, save_path)

print(f"saved: {save_path.resolve()}")
print("label counts -> fake(1):", int((label_tensor == 1).sum().item()) if label_tensor.numel() else 0, "real(0):", int((label_tensor == 0).sum().item()) if label_tensor.numel() else 0)

c:\Users\whyam\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\whyam\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\autograd\graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
c:\Users\whyam\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\whyam\AppData\Lo

Processed files: 16/400 | saved vectors so far: 4283
Processed files: 32/400 | saved vectors so far: 8096
Processed files: 48/400 | saved vectors so far: 11507
Processed files: 64/400 | saved vectors so far: 14379
Processed files: 80/400 | saved vectors so far: 17752
Processed files: 96/400 | saved vectors so far: 21567
Processed files: 112/400 | saved vectors so far: 24510
Processed files: 128/400 | saved vectors so far: 29281
Processed files: 144/400 | saved vectors so far: 31731
Processed files: 160/400 | saved vectors so far: 35532
Processed files: 176/400 | saved vectors so far: 40261
Processed files: 192/400 | saved vectors so far: 44531
Processed files: 208/400 | saved vectors so far: 47923
Processed files: 224/400 | saved vectors so far: 51753
Processed files: 240/400 | saved vectors so far: 55149
Processed files: 256/400 | saved vectors so far: 58533
Processed files: 272/400 | saved vectors so far: 62369
Processed files: 288/400 | saved vectors so far: 66644
Processed files: 3